# 05 SVM de régression (SVR) sur le score de compromis

Dans le notebook 02, nous avons défini la qualité d'une soudure par un **score continu** :

$$y = \sqrt{R_{\mathrm{UTS}} \times R_{\mathrm{allongement}}}$$

Nous prédisons ce score à partir de la composition et des paramètres de soudage avec une **machine à vecteurs de support en régression (SVR)**.

Nous respectons les règles posées dans les notebooks précédents :

- les **groupes** (essais d'un même dépôt) ne sont jamais séparés entre apprentissage et validation : `GroupKFold` ;
- dans **chaque pli**, la référence des rangs, donc **la cible**, est recalculée sur l'apprentissage seul (`calculer_cible`) ;
- dans **chaque pli**, la préparation de X (colonnes gardées, médiane, catégories, standardisation) est **réajustée** sur l'apprentissage seul (`creer_preparation`) ;
- UTS et allongement ne sont pas dans X ;
- le **test** n'est utilisé qu'une seule fois, à la fin ;
- les figures sont enregistrées dans `figures/`.

La standardisation est indispensable pour un SVR : sans elle, une variable en ppm écraserait une variable en % massique.

**Plan** : 1. Chargement · 2. Plis de validation · 3. Références simples · 4. Recherche d'hyperparamètres · 5. Évaluation sur le test · 6. Lecture du modèle · 7. Conclusion

## 1 Charger les données des notebooks 02 et 03

Nous reprenons `outputs/02/dataset_traction.csv` : les entrées X (non imputées), les mesures UTS et allongement, le score, les groupes et la partition développement/test. Nous ne tirons pas de nouveau test.

Si le fichier est absent, il faut d'abord exécuter le notebook 02.

In [1]:
from pathlib import Path
from functools import partial
from itertools import product
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FIGURES = ROOT / 'figures'
OUT = ROOT / 'outputs/05'
FIGURES.mkdir(exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# Les fonctions communes sont définies une seule fois dans src/.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.preparation import calculer_cible, creer_preparation

RANDOM_STATE = 42

ModuleNotFoundError: No module named 'src.preparation'

In [ ]:
fichier = ROOT / 'outputs/02/dataset_traction.csv'
assert fichier.exists(), "Exécuter d'abord le notebook 02 (outputs/02/dataset_traction.csv absent)."

categories = ['AC_DC', 'Electrode_pol', 'WeldType']
donnees = pd.read_csv(
    fichier, index_col='row_id',
    dtype={c: str for c in categories + ['WeldID', 'partition']},
    keep_default_na=False, na_values=[''], float_precision='round_trip'
)
schema = json.loads((ROOT / 'data/processed/welddb_metadata.json').read_text())['schema']
variables_X = schema['COMPOSITION'] + schema['PROCESS']
mesures = ['UTS_MPa', 'Elongation_pct']

assert not set(variables_X) & set(mesures + ['score_compromis', 'WeldID', 'groupe'])

dev = donnees['partition'] == 'developpement'
test = donnees['partition'] == 'test'
assert (dev | test).all()

X_dev, X_test = donnees.loc[dev, variables_X], donnees.loc[test, variables_X]
mesures_dev, mesures_test = donnees.loc[dev, mesures], donnees.loc[test, mesures]
groupes_dev = donnees.loc[dev, 'groupe']
y_dev, y_test = donnees.loc[dev, 'score_compromis'], donnees.loc[test, 'score_compromis']

assert set(groupes_dev).isdisjoint(set(donnees.loc[test, 'groupe']))
print(f'Développement : {len(X_dev)} lignes, {groupes_dev.nunique()} groupes.')
print(f'Test réservé  : {len(X_test)} lignes.')
print(f'Entrées : {X_dev.shape[1]} (avant sélection et encodage).')

In [ ]:
# Contrôle de cohérence : le score exporté sur le développement doit être celui que recalcule calculer_cible.
recalcule = calculer_cible(mesures_dev, groupes_dev, mesures_dev)
ecart = float(np.max(np.abs(recalcule.to_numpy() - y_dev.to_numpy())))
print(f'Écart maximal entre score exporté et score recalculé : {ecart:.2e}')
if ecart > 1e-9:
    print("Attention : le score exporté diffère du score recalculé. Vérifier src/preparation.py et le notebook 02.")

## 2 Construire les plis de validation

Cinq plis `GroupKFold`. Pour chaque pli, nous faisons trois choses, **uniquement avec l'apprentissage** :

1. calculer la référence des rangs et en déduire les scores d'apprentissage ;
2. appliquer cette même référence à la validation ;
3. ajuster la préparation de X, puis la transformer sur l'apprentissage et la validation.

Ces trois étapes ne dépendent pas des hyperparamètres du SVR : nous les faisons **une seule fois** et nous réutilisons les plis pour tous les modèles.

In [ ]:
validation = GroupKFold(n_splits=5)
plis = []

for numero, (train, val) in enumerate(validation.split(X_dev, groups=groupes_dev), start=1):
    ids_tr, ids_va = X_dev.index[train], X_dev.index[val]
    assert set(groupes_dev.loc[ids_tr]).isdisjoint(set(groupes_dev.loc[ids_va]))

    y_tr = calculer_cible(mesures_dev.loc[ids_tr], groupes_dev.loc[ids_tr], mesures_dev.loc[ids_tr])
    y_va = calculer_cible(mesures_dev.loc[ids_tr], groupes_dev.loc[ids_tr], mesures_dev.loc[ids_va])

    preparation_pli, gardees_pli = creer_preparation(X_dev.loc[ids_tr])
    X_tr = np.asarray(preparation_pli.fit_transform(X_dev.loc[ids_tr]), dtype=float)
    X_va = np.asarray(preparation_pli.transform(X_dev.loc[ids_va]), dtype=float)
    assert np.isfinite(X_tr).all() and np.isfinite(X_va).all()

    plis.append({'X_tr': X_tr, 'X_va': X_va,
                 'y_tr': y_tr.to_numpy(), 'y_va': y_va.to_numpy()})

pd.DataFrame([{'pli': i + 1, 'apprentissage': len(p['y_tr']), 'validation': len(p['y_va']),
               'colonnes': p['X_tr'].shape[1]} for i, p in enumerate(plis)])

## 3 Références simples

Un SVR n'a de valeur que s'il fait mieux que des modèles triviaux. Nous mesurons :

- un **modèle constant** qui prédit toujours la moyenne du score (un R² proche de 0 est attendu) ;
- une **régression linéaire régularisée** (Ridge).

Mesures : **R²** (part de variance expliquée), **RMSE** et **MAE** (erreurs en unités de score, entre 0 et 1) et **corrélation de Spearman** (le classement des soudures est-il bien restitué ? utile ici car le score est construit sur des rangs).

In [ ]:
def metriques(y_vrai, y_pred):
    return {'R2': r2_score(y_vrai, y_pred),
            'RMSE': float(np.sqrt(mean_squared_error(y_vrai, y_pred))),
            'MAE': mean_absolute_error(y_vrai, y_pred),
            # Spearman n'est pas défini pour une prédiction constante (modèle de référence)
            'Spearman': spearmanr(y_vrai, y_pred)[0] if np.ptp(y_pred) > 0 else np.nan}


def evaluer_cv(fabrique):
    # Entraîne un modèle neuf sur chaque pli et renvoie les métriques de validation.
    lignes = []
    for p in plis:
        modele = fabrique().fit(p['X_tr'], p['y_tr'])
        lignes.append(metriques(p['y_va'], modele.predict(p['X_va'])))
    return pd.DataFrame(lignes)


def resume(nom, tableau):
    moyenne, ecart = tableau.mean(), tableau.std()
    return {'modèle': nom, **{f'{m}': f'{moyenne[m]:.3f} ± {ecart[m]:.3f}' for m in tableau.columns}}


cv_constant = evaluer_cv(DummyRegressor)
cv_ridge = evaluer_cv(partial(Ridge, alpha=1.0))
pd.DataFrame([resume('Constant (moyenne)', cv_constant), resume('Ridge', cv_ridge)]).set_index('modèle')

## 4 Recherche d'hyperparamètres du SVR

Trois réglages comptent :

| Paramètre | Rôle |
|---|---|
| `C` | pénalité des erreurs : grand = suit finement les données (risque de surapprentissage), petit = modèle plus lisse |
| `epsilon` | largeur du « tube » sans pénalité autour de la prédiction (le score est entre 0 et 1, d'où des valeurs petites) |
| `gamma` | portée du noyau RBF : grand = influence très locale de chaque point |

Nous comparons le noyau **RBF** (non linéaire) et le noyau **linéaire**. Chaque combinaison est évaluée sur les mêmes cinq plis ; nous retenons celle qui minimise la **RMSE** moyenne de validation.

Le test n'intervient pas dans ce choix.

In [ ]:
grille = []
for C, eps, gamma in product([0.1, 1, 10, 100], [0.01, 0.05, 0.1], ['scale', 0.003, 0.01, 0.03]):
    grille.append({'kernel': 'rbf', 'C': C, 'epsilon': eps, 'gamma': gamma})
for C, eps in product([0.01, 0.1, 1, 10], [0.01, 0.05, 0.1]):
    grille.append({'kernel': 'linear', 'C': C, 'epsilon': eps, 'gamma': 'scale'})

resultats = []
for params in grille:
    tableau = evaluer_cv(partial(SVR, **params))
    resultats.append({**params, 'RMSE': tableau['RMSE'].mean(), 'R2': tableau['R2'].mean(),
                      'Spearman': tableau['Spearman'].mean()})

resultats = pd.DataFrame(resultats).sort_values('RMSE').reset_index(drop=True)
resultats.to_csv(OUT / 'svr_grille_validation_croisee.csv', index=False)
print(f'{len(resultats)} combinaisons évaluées. Les 8 meilleures :')
display(resultats.head(8).round(3))

meilleurs = resultats.iloc[0][['kernel', 'C', 'epsilon', 'gamma']].to_dict()
meilleurs['C'] = float(meilleurs['C'])
meilleurs['epsilon'] = float(meilleurs['epsilon'])
if meilleurs['gamma'] != 'scale':
    meilleurs['gamma'] = float(meilleurs['gamma'])
print('Hyperparamètres retenus :', meilleurs)

In [ ]:
# Sensibilité au couple (C, gamma) pour le noyau RBF, à epsilon fixé au meilleur choix.
rbf = resultats[(resultats['kernel'] == 'rbf') & (resultats['epsilon'].astype(float) == (meilleurs['epsilon']))].copy()
rbf['gamma'] = rbf['gamma'].astype(str)
tab = rbf.pivot_table(index='C', columns='gamma', values='R2', aggfunc='max')

fig, ax = plt.subplots(figsize=(6.5, 4.5))
image = ax.imshow(tab.values, cmap='viridis', aspect='auto')
ax.set_xticks(range(tab.shape[1]))
ax.set_xticklabels(tab.columns)
ax.set_yticks(range(tab.shape[0]))
ax.set_yticklabels(tab.index)
ax.set_xlabel('gamma')
ax.set_ylabel('C')
ax.set_title(f"R² en validation croisée (RBF, epsilon = {meilleurs['epsilon']})")
for i in range(tab.shape[0]):
    for j in range(tab.shape[1]):
        ax.text(j, i, f'{tab.values[i, j]:.2f}', ha='center', va='center', color='white', fontsize=9)
fig.colorbar(image, label='R² moyen')
fig.tight_layout()
fig.savefig(FIGURES / 'svr_sensibilite_C_gamma.png', dpi=150)
plt.show()

In [ ]:
# Performance en validation croisée du SVR retenu, à côté des références.
cv_svr = evaluer_cv(partial(SVR, **meilleurs))
tableau_cv = pd.DataFrame([resume('Constant (moyenne)', cv_constant),
                           resume('Ridge', cv_ridge),
                           resume('SVR', cv_svr)]).set_index('modèle')
tableau_cv.to_csv(OUT / 'svr_validation_croisee.csv')
tableau_cv

## 5 Évaluation finale sur le test

Le test n'a servi ni à calculer la référence des rangs, ni à ajuster la préparation, ni à choisir les hyperparamètres. Nous l'utilisons **une seule fois**.

Pour l'évaluation finale :

- la **référence des rangs** est celle de tout le développement (c'est le score exporté par le notebook 02, appliqué au test) ;
- la **préparation** est réajustée sur tout le développement ;
- le SVR est entraîné sur tout le développement, puis évalué sur le test.

In [ ]:
preparation, gardees = creer_preparation(X_dev)
X_dev_p = np.asarray(preparation.fit_transform(X_dev), dtype=float)
X_test_p = np.asarray(preparation.transform(X_test), dtype=float)
noms_colonnes = list(preparation.get_feature_names_out())
assert np.isfinite(X_dev_p).all() and np.isfinite(X_test_p).all()

modeles = {'Constant (moyenne)': DummyRegressor(),
           'Ridge': Ridge(alpha=1.0),
           'SVR': SVR(**meilleurs)}
lignes = []
for nom, modele in modeles.items():
    modele.fit(X_dev_p, y_dev.to_numpy())
    lignes.append({'modèle': nom, **metriques(y_test.to_numpy(), modele.predict(X_test_p))})

resultat_test = pd.DataFrame(lignes).set_index('modèle')
resultat_test.to_csv(OUT / 'svr_resultats_test.csv')
resultat_test.round(3)

In [ ]:
svr = modeles['SVR']
y_pred = svr.predict(X_test_p)
residus = y_test.to_numpy() - y_pred

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].scatter(y_test, y_pred, s=25, alpha=0.7)
axes[0].plot([0, 1], [0, 1], color='black', linestyle='--', linewidth=1, label='prédiction parfaite')
axes[0].set_xlabel('Score réel')
axes[0].set_ylabel('Score prédit')
axes[0].set_title('SVR : prédiction contre réalité (test)')
axes[0].legend()

axes[1].hist(residus, bins=20, edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Erreur (réel - prédit)')
axes[1].set_ylabel('Nombre de soudures')
axes[1].set_title('Distribution des erreurs (test)')

fig.tight_layout()
fig.savefig(FIGURES / 'svr_prediction_test.png', dpi=150)
plt.show()

## 6 Lire le modèle : quelles entrées comptent ?

Un SVR à noyau RBF n'a pas de coefficients lisibles. Nous mesurons l'**importance par permutation** : on mélange au hasard une colonne du test et on regarde de combien le R² baisse. Plus la baisse est grande, plus le modèle s'appuie sur cette colonne.

Cette lecture décrit **le modèle**, pas une loi physique : deux colonnes corrélées peuvent se partager l'importance.

In [ ]:
importance = permutation_importance(svr, X_test_p, y_test.to_numpy(), n_repeats=20,
                                    scoring='r2', random_state=RANDOM_STATE)
imp = pd.DataFrame({'colonne': noms_colonnes, 'baisse_R2': importance.importances_mean,
                    'ecart_type': importance.importances_std}).sort_values('baisse_R2', ascending=False)
imp.to_csv(OUT / 'svr_importance_permutation.csv', index=False)

top = imp.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top['colonne'], top['baisse_R2'], xerr=top['ecart_type'])
ax.set_xlabel('Baisse du R² quand la colonne est mélangée')
ax.set_title('Importance par permutation (SVR, test)')
fig.tight_layout()
fig.savefig(FIGURES / 'svr_importance_permutation.png', dpi=150)
plt.show()

## 7 Conclusion

Les tableaux ci-dessus donnent les chiffres à reporter dans le rapport :

- `outputs/05/svr_validation_croisee.csv` : performances en validation croisée groupée (référence et cible recalculées dans chaque pli) ;
- `outputs/05/svr_resultats_test.csv` : performances finales sur le test, comparées au modèle constant et à Ridge ;
- `figures/svr_*.png` : sensibilité à `C` et `gamma`, prédiction contre réalité, importances.

**Points à discuter dans le rapport**

- Le SVR est-il meilleur que Ridge ? Si l'écart est faible, la relation composition → score est peu non linéaire, ou les données sont trop peu nombreuses (666 lignes, moins de groupes) pour en tirer parti.
- Le **test est petit** (une centaine de lignes, quelques dizaines de groupes) : l'incertitude sur R² est grande. Nous comparons à l'écart-type entre plis.
- Le score est **relatif au corpus** (rangs). Un score prédit élevé signifie « bon compromis par rapport aux soudures de cette base », pas une conformité industrielle.
- L'imputation par la **médiane** est un choix de départ : 25 % de remplissage minimum signifie que certaines colonnes sont remplies à 70 % par une constante.

**Pistes**

- comparer d'autres seuils de remplissage et une imputation par régression ou k plus proches voisins ;
- tester d'autres poids dans le score (notebook 02) et regarder si le classement des modèles change ;
- comparer avec les autres modèles supervisés de l'équipe sur **les mêmes plis** et le même test ;
- version classification du SVM : découper le score en classes (par exemple tiers inférieur / supérieur) avec des seuils justifiés.

> Avant de committer : `Kernel > Restart & Clear All Outputs`.